In [1]:
import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import fsolve, minimize

N = stats.norm
RNG = np.random.default_rng(42)


def header(title):
    print("\n" + "=" * 72 + f"\n{title}\n" + "=" * 72)


# =====================================================================
# ТЕМА 2. Волатильность, распределения доходностей, VaR, ES
# =====================================================================
def var_es_hist(returns, conf=0.99):
    loss = -np.asarray(returns)
    var = np.percentile(loss, conf * 100)
    return var, loss[loss >= var].mean()


def var_es_normal(returns, conf=0.99):
    mu, sd = np.mean(returns), np.std(returns, ddof=1)
    z = N.ppf(conf)
    return -mu + z * sd, -mu + sd * N.pdf(z) / (1 - conf)


def topic2():
    header("ТЕМА 2. VaR и ES: толстые хвосты против нормального распределения")
    nu = 4
    r = 0.01 * RNG.standard_t(nu, 5000) / np.sqrt(nu / (nu - 2))  # 1% в день, толстые хвосты
    print(f"Эксцесс: {stats.kurtosis(r):.2f} (у нормального 0), Харке-Бера p = {stats.jarque_bera(r).pvalue:.1e}")
    for conf in (0.95, 0.99, 0.999):
        h, he = var_es_hist(r, conf)
        p, pe = var_es_normal(r, conf)
        print(f"conf={conf}: VaR hist {h:.4f} | VaR normal {p:.4f} | ES hist {he:.4f} | ES normal {pe:.4f}")
    print("Вывод: чем выше уровень доверия, тем сильнее нормальное распределение занижает хвост.")
    # EWMA волатильность
    lam = 0.94
    var_t = r[:30].var()
    for x in r[30:]:
        var_t = lam * var_t + (1 - lam) * x ** 2
    print(f"EWMA-волатильность на последний день: {np.sqrt(var_t):.4f}, простая за всё окно: {r.std():.4f}")
    return r


# =====================================================================
# ТЕМА 3. Стресс-тесты, сценарии, отображение на факторы риска
# =====================================================================
def topic3():
    header("ТЕМА 3. Отображение позиций на факторы риска и стресс-сценарии")
    equity, bond, usd = 50e6, 30e6, 20e6            # позиции, $
    beta, duration = 1.1, 5.0                       # бета акций к индексу, дюрация облигаций
    dv01 = bond * duration * 0.0001                 # потеря при росте ставки на 1 б.п.
    print(f"DV01 облигаций: {dv01:,.0f} $ на 1 б.п.")
    scenarios = {
        "Акции -20%":                dict(idx=-0.20, rate_bp=0,   usd=0.0),
        "Ставки +100 б.п.":          dict(idx=0.0,   rate_bp=100, usd=0.0),
        "Курс USD +20% (тенге -)":   dict(idx=0.0,   rate_bp=0,   usd=0.20),
        "Комбинированный кризис":    dict(idx=-0.30, rate_bp=150, usd=0.15),
    }
    rows = []
    for name, s in scenarios.items():
        pnl = equity * beta * s["idx"] - dv01 * s["rate_bp"] + usd * s["usd"]
        rows.append((name, pnl))
    print(pd.DataFrame(rows, columns=["Сценарий", "PnL, $"]).to_string(index=False, float_format=lambda x: f"{x:,.0f}"))
    print("Важно: в одном сценарии факторы двигаются вместе, это и отличает его от анализа чувствительности.")


# =====================================================================
# ТЕМА 4. Кредитный риск: модель Мертона, PD, CVA
# =====================================================================
def merton_equity(V, D, T, r, sig):
    d1 = (np.log(V / D) + (r + 0.5 * sig ** 2) * T) / (sig * np.sqrt(T))
    d2 = d1 - sig * np.sqrt(T)
    return V * N.cdf(d1) - D * np.exp(-r * T) * N.cdf(d2), d1, d2


def topic4():
    header("ТЕМА 4. Кредитный риск: модель Мертона и CVA")
    E, sig_E, D, T, r = 40.0, 0.50, 60.0, 1.0, 0.03   # капитализация, её волатильность, долг, срок, ставка

        def eqs(x):
        V, sig_V = x
        Eh, d1, _ = merton_equity(V, D, T, r, sig_V)
        return [Eh - E, N.cdf(d1) * sig_V * V - sig_E * E]

    V, sig_V = fsolve(eqs, [E + D, 0.2])
    _, d1, d2 = merton_equity(V, D, T, r, sig_V)
    pd_ = N.cdf(-d2)
    print(f"Стоимость активов V = {V:.2f}, волатильность активов = {sig_V:.3f}")
    print(f"Distance to default (d2) = {d2:.2f}, PD за год (риск-нейтральная) = {pd_:.2%}")
    # стоимость рискового долга B = V - E = D*exp(-rT)*N(d2) + V*N(-d1); доходность y = -ln(B/D)/T; спред = y - r
    B = V - E
    spread = -np.log(B / D) / T - r
    spread_formula = -np.log(N.cdf(d2) + V / (D * np.exp(-r * T)) * N.cdf(-d1)) / T
    print(f"Стоимость рискового долга B = {B:.2f} (безрисковая стоимость долга {D * np.exp(-r * T):.2f})")
    print(f"Кредитный спред по модели = {spread * 1e4:.1f} б.п. (по формуле {spread_formula * 1e4:.1f} б.п.)")
    # CVA контракта-форварда
    S0, vol, Tc, R = 100.0, 0.25, 2.0, 0.40
    lam = 0.02 / (1 - R)                              # интенсивность дефолта из спреда 2%
    t = np.linspace(0.25, Tc, 8)
    K = S0 * np.exp(r * Tc)
    n = 100_000
    z = RNG.standard_normal(n)
    cva, prev_pd = 0.0, 0.0
    for ti in t:
        St = S0 * np.exp((r - 0.5 * vol ** 2) * ti + vol * np.sqrt(ti) * z)
        mtm = St - K * np.exp(-r * (Tc - ti))
        ee = np.maximum(mtm, 0).mean()
        cur_pd = 1 - np.exp(-lam * ti)
        cva += (1 - R) * np.exp(-r * ti) * ee * (cur_pd - prev_pd)
        prev_pd = cur_pd
    print(f"CVA форварда на 2 года (спред контрагента 2%, recovery 40%) ~ {cva:.3f} на 100 номинала")
    return d2, pd_, spread, spread_formula


# =====================================================================
# ТЕМА 5. Опционы: Black-Scholes, греки, биномиальное дерево
# =====================================================================
def bs_d(S, K, T, r, sig, q=0.0):
    d1 = (np.log(S / K) + (r - q + 0.5 * sig ** 2) * T) / (sig * np.sqrt(T))
    return d1, d1 - sig * np.sqrt(T)


def bs_price(S, K, T, r, sig, q=0.0, kind="call"):
    d1, d2 = bs_d(S, K, T, r, sig, q)
    if kind == "call":
        return S * np.exp(-q * T) * N.cdf(d1) - K * np.exp(-r * T) * N.cdf(d2)
    return K * np.exp(-r * T) * N.cdf(-d2) - S * np.exp(-q * T) * N.cdf(-d1)


def bs_greeks(S, K, T, r, sig, q=0.0, kind="call"):
    d1, d2 = bs_d(S, K, T, r, sig, q)
    pdf = N.pdf(d1)
    sign = 1 if kind == "call" else -1
    delta = np.exp(-q * T) * (N.cdf(d1) if kind == "call" else N.cdf(d1) - 1)
    gamma = np.exp(-q * T) * pdf / (S * sig * np.sqrt(T))
    vega = S * np.exp(-q * T) * pdf * np.sqrt(T)
    theta = (-S * np.exp(-q * T) * pdf * sig / (2 * np.sqrt(T))
             - sign * r * K * np.exp(-r * T) * N.cdf(sign * d2)
             + sign * q * S * np.exp(-q * T) * N.cdf(sign * d1))
    rho = sign * K * T * np.exp(-r * T) * N.cdf(sign * d2)
    return dict(delta=delta, gamma=gamma, vega=vega, theta=theta, rho=rho)


def binomial(S, K, T, r, sig, n=500, q=0.0, kind="call", american=False):
    dt = T / n
    u = np.exp(sig * np.sqrt(dt))
    d = 1 / u
    p = (np.exp((r - q) * dt) - d) / (u - d)
    disc = np.exp(-r * dt)
    j = np.arange(n + 1)
    ST = S * u  j * d  (n - j)
    V = np.maximum(ST - K, 0) if kind == "call" else np.maximum(K - ST, 0)
    for i in range(n - 1, -1, -1):
        V = disc * (p * V[1:i + 2] + (1 - p) * V[0:i + 1])
        if american:
            jj = np.arange(i + 1)
            Sj = S * u  jj * d  (i - jj)
            V = np.maximum(V, np.maximum(Sj - K, 0) if kind == "call" else np.maximum(K - Sj, 0))
    return V[0]


def topic5():
    header("ТЕМА 5. Black-Scholes, греки и биномиальная модель")
    S, K, T, r, sig = 100.0, 100.0, 1.0, 0.05, 0.20
    c, p = bs_price(S, K, T, r, sig), bs_price(S, K, T, r, sig, kind="put")
    g = bs_greeks(S, K, T, r, sig)
    print(f"Call = {c:.4f}, Put = {p:.4f}")
    print("Паритет call-put: C - P = S - K*exp(-rT):", round(c - p, 6), "=", round(S - K * np.exp(-r * T), 6))


print(f"Греки колла: delta={g['delta']:.4f}, gamma={g['gamma']:.5f}, vega(на 1 п.п. vol)={g['vega'] / 100:.4f}, "
          f"theta(в день)={g['theta'] / 365:.4f}, rho(на 1 п.п. ставки)={g['rho'] / 100:.4f}")
    print("Сходимость биномиального дерева к BS (европейский колл):")
    for n in (10, 50, 200, 1000):
        print(f"  n={n:5d}: {binomial(S, K, T, r, sig, n):.4f}")
    am = binomial(S, K, T, r, sig, 500, kind="put", american=True)
    eu = binomial(S, K, T, r, sig, 500, kind="put")
    print(f"Американский put = {am:.4f} > европейский put = {eu:.4f} (премия за досрочное исполнение {am - eu:.4f})")
    return c, p, g


# =====================================================================
# ТЕМА 6. Хеджирование: ошибка дельта-хеджа при разной частоте ребалансировки
# =====================================================================
def delta_hedge_error(S0, K, T, r, sig, mu, n_steps, n_paths=20_000):
    dt = T / n_steps
    z = RNG.standard_normal((n_paths, n_steps))
    S = np.empty((n_paths, n_steps + 1))
    S[:, 0] = S0
    S[:, 1:] = S0 * np.exp(np.cumsum((mu - 0.5 * sig ** 2) * dt + sig * np.sqrt(dt) * z, axis=1))
    c0 = bs_price(S0, K, T, r, sig)
    delta_prev = bs_greeks(S0, K, T, r, sig)["delta"] * np.ones(n_paths)
    cash = c0 - delta_prev * S0                                    # продали колл, купили дельту акций
    for i in range(1, n_steps):
        tau = T - i * dt
        delta = bs_greeks(S[:, i], K, tau, r, sig)["delta"]
        cash = cash * np.exp(r * dt) - (delta - delta_prev) * S[:, i]
        delta_prev = delta
    cash = cash * np.exp(r * dt) + delta_prev * S[:, -1] - np.maximum(S[:, -1] - K, 0)
    return cash


def topic6():
    header("ТЕМА 6. Дельта-хеджирование проданного колла")
    out = []
    for n in (4, 12, 52, 252):
        e = delta_hedge_error(100, 100, 1.0, 0.05, 0.20, 0.10, n)
        out.append((n, e.mean(), e.std()))
        print(f"Ребалансировок: {n:4d} | средняя ошибка {e.mean():+.4f} | разброс (std) {e.std():.4f}")
    print("Вывод: ошибка хеджа падает примерно как 1/sqrt(числа ребалансировок), но не исчезает (транзакционные издержки в симуляции не учтены).")
    return out


# =====================================================================
# ТЕМА 7. Оптимизация портфеля: минимальная дисперсия, максимум Шарпа, risk parity
# =====================================================================
def portfolio_inputs():
    mu = np.array([0.08, 0.10, 0.07, 0.03, 0.05])
    vol = np.array([0.20, 0.25, 0.15, 0.08, 0.06])
    corr = np.array([[1.0, 0.6, 0.5, 0.1, 0.0],
                     [0.6, 1.0, 0.4, 0.1, 0.0],
                     [0.5, 0.4, 1.0, 0.2, 0.1],
                     [0.1, 0.1, 0.2, 1.0, 0.5],
                     [0.0, 0.0, 0.1, 0.5, 1.0]])
    return mu, np.outer(vol, vol) * corr


def optimize_weights(objective, n):
    cons = ({"type": "eq", "fun": lambda w: w.sum() - 1},)
    res = minimize(objective, np.ones(n) / n, method="SLSQP", bounds=[(0, 1)] * n, constraints=cons,
                   options={"ftol": 1e-12, "maxiter": 500})
    return res.x


def topic7():
    header("ТЕМА 7. Марковиц и risk parity")
    mu, cov = portfolio_inputs()
    n, rf = len(mu), 0.03
    names = ["Акции KZ", "Акции EM", "Акции US", "Облиг. краткие", "Облиг. длинные"]
    vol_p = lambda w: np.sqrt(w @ cov @ w)
    w_mv = optimize_weights(lambda w: w @ cov @ w, n)
    w_ms = optimize_weights(lambda w: -(w @ mu - rf) / vol_p(w), n)

    def rp_obj(w):
        rc = w * (cov @ w) / vol_p(w)
        return ((rc - vol_p(w) / n) ** 2).sum()

    w_rp = optimize_weights(rp_obj, n)
    table = pd.DataFrame({"Мин. дисперсия": w_mv, "Макс. Шарп": w_ms, "Risk parity": w_rp}, index=names)
    print((table * 100).round(1).to_string())
    stats_tbl = pd.DataFrame({
        "Доходность": [w @ mu for w in (w_mv, w_ms, w_rp)],


"Волатильность": [vol_p(w) for w in (w_mv, w_ms, w_rp)],
        "Шарп": [(w @ mu - rf) / vol_p(w) for w in (w_mv, w_ms, w_rp)],
    }, index=table.columns)
    print(stats_tbl.round(4).to_string())
    rc = w_rp * (cov @ w_rp) / vol_p(w_rp)
    print("Вклады в риск при risk parity (должны быть равны):", np.round(rc / rc.sum(), 3))
    return w_mv, w_ms, w_rp


# =====================================================================
# ТЕМА 8. Монте-Карло, стохастические процессы, GARCH(1,1)
# =====================================================================
def mc_call_price(S0, K, T, r, sig, n=200_000):
    z = RNG.standard_normal(n // 2)
    z = np.concatenate([z, -z])                                    # антитетические переменные
    ST = S0 * np.exp((r - 0.5 * sig ** 2) * T + sig * np.sqrt(T) * z)
    pay = np.exp(-r * T) * np.maximum(ST - K, 0)
    return pay.mean(), pay.std(ddof=1) / np.sqrt(len(pay))


def simulate_garch(n, omega, alpha, beta):
    r = np.zeros(n)
    h = omega / (1 - alpha - beta)
    for t in range(n):
        r[t] = np.sqrt(h) * RNG.standard_normal()
        h = omega + alpha * r[t] ** 2 + beta * h
    return r


def garch_nll(params, r):
    omega, alpha, beta = params
    h = r.var()
    ll = 0.0
    for x in r:
        ll += 0.5 * (np.log(h) + x ** 2 / h)
        h = omega + alpha * x ** 2 + beta * h
    return ll


def fit_garch(r):
    res = minimize(garch_nll, x0=[0.05, 0.05, 0.85], args=(r,), method="SLSQP",
                   bounds=[(1e-6, 5), (1e-4, 0.5), (1e-4, 0.999)],
                   constraints=({"type": "ineq", "fun": lambda p: 0.999 - p[1] - p[2]},))
    omega, alpha, beta = res.x
    h = r.var()
    for x in r:
        h = omega + alpha * x ** 2 + beta * h                      # прогноз дисперсии на следующий день
    return res.x, h


def topic8():
    header("ТЕМА 8. Монте-Карло и GARCH(1,1)")
    price, se = mc_call_price(100, 100, 1.0, 0.05, 0.20)
    bs = bs_price(100, 100, 1.0, 0.05, 0.20)
    print(f"Монте-Карло колл = {price:.4f} (станд. ошибка {se:.4f}) | Black-Scholes = {bs:.4f} | разница {abs(price - bs) / se:.1f} ст. ошибок")
    true = (0.02, 0.08, 0.90)                                      # доходности в процентах
    r = simulate_garch(2500, *true)
    est, h_next = fit_garch(r)
    print(f"GARCH истинные параметры (omega, alpha, beta) = {true}")
    print(f"GARCH оценка (одна выборка, поэтому не точно)   = {tuple(float(x) for x in np.round(est, 3))}")
    sd_next = np.sqrt(h_next)
    print(f"Прогноз волатильности на завтра = {sd_next:.3f}% ; 1-дневный VaR 99% = {N.ppf(0.99) * sd_next:.3f}% портфеля")
    print("Персистентность alpha+beta =", round(est[1] + est[2], 3), "(близко к 1: шоки волатильности затухают медленно)")
    return est, price, se, bs


# =====================================================================
# САМОПРОВЕРКИ: если что-то из этого падает, значит, ошибка в коде
# =====================================================================
def self_checks(c, p, g, price, se, bs, est, d2, pd_, spread, spread_formula):
    header("САМОПРОВЕРКИ")
    checks = {
        "Black-Scholes call = 10.4506 (эталон)": abs(c - 10.4506) < 1e-3,
        "Black-Scholes put = 5.5735 (эталон)": abs(p - 5.5735) < 1e-3,
        "Паритет call-put": abs((c - p) - (100 - 100 * np.exp(-0.05))) < 1e-9,
        "delta колла = 0.6368 (эталон)": abs(g["delta"] - 0.6368) < 1e-3,
        "Биномиальное дерево ~ BS (n=2000)": abs(binomial(100, 100, 1, 0.05, 0.2, 2000) - c) < 0.01,
        "Монте-Карло в пределах 3 ст. ошибок от BS": abs(price - bs) < 3 * se,
        "GARCH: alpha+beta в разумных пределах (0.85-0.995)": 0.85 < est[1] + est[2] < 0.995,
        "Мертон: PD = N(-d2)": abs(pd_ - N.cdf(-d2)) < 1e-12,
        "Мертон: спред положительный": spread > 0,
        "Мертон: спред через V-E совпадает со спредом по формуле": abs(spread - spread_formula) < 1e-4,
    }
    for name, ok in checks.items():
        print(("OK   " if ok else "FAIL ") + name)
    assert all(checks.values()), "Есть непройденные самопроверки"


def main():
    topic2()
    topic3()
    d2, pd_, spread, spread_formula = topic4()
    c, p, g = topic5()
    topic6()
    topic7()
    est, price, se, bs = topic8()
    self_checks(c, p, g, price, se, bs, est, d2, pd_, spread, spread_formula)


if name == "main":
    main()

IndentationError: unexpected indent (26773537.py, line 84)